<a href="https://colab.research.google.com/github/derekhatherley/ds2002-fa26/blob/main/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [10]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [11]:
df['revenue'] = df['qty']* df['price']
total_revenue = df['revenue'].sum()
print('The total revenue is: $', total_revenue)

The total revenue is: $ 8520.0


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [12]:
food_rev = df[df['category'] == 'Food']['revenue'].sum()
merch_rev = df[df['category'] == 'Merch']['revenue'].sum()
rain_rev = df[df['category'] == 'RainGear']['revenue'].sum()
drink_rev = df[df['category'] == 'Drink']['revenue'].sum()

by_category = pd.DataFrame({
    'category': ['Food', 'Merch', 'RainGear', 'Drink'],
    'revenue': [food_rev, merch_rev, rain_rev, drink_rev],
    'share (%)': [100*(food_rev/total_revenue), 100*(merch_rev/total_revenue), 100*(rain_rev/total_revenue), 100*(drink_rev/total_revenue)]
})

by_category.sort_values(by='revenue', ascending=False)

,category,revenue,share (%)
0,Food,4293.0,50.387324
1,Merch,1771.5,20.792254
3,Drink,1554.0,18.239437
2,RainGear,901.5,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [13]:
# TODO
v10_rev = df[df['vendor_id'] == 'V-10']['revenue'].sum()
v18_rev = df[df['vendor_id'] == 'V-18']['revenue'].sum()
v01_rev = df[df['vendor_id'] == 'V-01']['revenue'].sum()
v05_rev = df[df['vendor_id'] == 'V-05']['revenue'].sum()

v10_orders = df[df['vendor_id'] == 'V-10']['qty'].sum()
v18_orders = df[df['vendor_id'] == 'V-18']['qty'].sum()
v01_orders = df[df['vendor_id'] == 'V-01']['qty'].sum()
v05_orders = df[df['vendor_id'] == 'V-05']['qty'].sum()

v10_avg = v10_rev/v10_orders
v18_avg = v18_rev/v18_orders
v01_avg = v01_rev/v01_orders
v05_avg = v05_rev/v05_orders

avg_df = pd.DataFrame({
    'vendor': ['V-10', 'V-18', 'V-01', 'V-05'],
    'average order revenue': [v10_avg, v18_avg, v01_avg, v05_avg],
    'order count': [v10_orders, v18_orders, v01_orders, v05_orders]
})

avg_df.sort_values(by='average order revenue', ascending=False)


,vendor,average order revenue,order count
2,V-01,11.297872,188
1,V-18,10.824885,217
3,V-05,10.752809,178
0,V-10,10.665000,200


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [14]:
# TODO
print('The share of revenue coming from merch is:' , round(100*(merch_rev/total_revenue), 1) , '%')

The share of revenue coming from merch is: 20.8 %


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [15]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = pd.merge(df, vendor_names, on = 'vendor_id', how = 'left')

assert len(df) == len(joined)
# Length of tables did not change

joined['revenue'] = joined['qty']* joined['price']
joined_total_revenue = joined['revenue'].sum()
assert total_revenue == joined_total_revenue
# Revenue did not change

joined['vendor_name'] = joined['vendor_name'].fillna('Unknwon Vendor Name')
# Left NA vendor name in the table, but made the vendor name 'Unknown Vendor Name'

print('Checks passed. Total revenue and row number did not change.')
joined.head()

# TODO: merge, validate, and report the unmatched vendor

Checks passed. Total revenue and row number did not change.


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknwon Vendor Name
2,V-18,Drink,3,4.5,13.5,Unknwon Vendor Name
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknwon Vendor Name


**The unmatched vendor, and what I did about it:** _V-18 was the unmatched vendor. I left those rows in the table, but made the vendor name 'Unknown Vendor Name'._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [16]:
# TODO
pivot = joined.pivot_table(
    index = 'vendor_name',
    columns = 'category',
    values = 'revenue',
    aggfunc = 'sum',
    margins = True,
    margins_name = 'Total'
)
pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknwon Vendor Name,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [17]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) For the next game, I would strongly suggest that Hoos Burgers and Rotunda Tacos start to bundle their food and drink options. Starting with Hoos Burgers, they amassed 1338 dollars in food sales, yet only made 171 dollars from drinks. By adding a 'meal' option that includes a drink with a food item for only a dollar or two more, the revenue associated with food sales would increase without sacrificing much in terms of drink revenue since it was so low to begin with. Rotunda Tacos can also implement a bundle such as this, since their revenue is also dominated by food orders, at 882 dollars, while only amassing 298.50 dollars in drink revenue. A bundle would increase the revenue for these vendors without needing more customers, since food revenue is already high.

b) The least truthworthy answer is Question 5. The fact that there is a vendor id that is missing a match detracts from the validity of the revenue calculations for each vendor. I made the decision to leave the rows with an unmatched vendor in the table, but that could impact the overall data in multiple ways. For example, vendor 18 could be an outdated ID or a typo that really should have its revenue be assigned to one of the matched vendors. This data table gives us no way to know why there is no match, which makes its interpretation untrustworthy.